In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
import re
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

In [2]:
data = pd.read_csv(r'C:\Users\LN\Documents\Jedha\Projets\NORTH_FACE\Helene\sample_clean.csv')

In [3]:
data.head()

,id,description_text,details,fabric,weight,pays
0,1,active classic boxers - there s a reason why o...,"silky capilene 1 fabric is ultralight, breatha...",3.7-oz 100% all-recycled polyester with gladio...,99 g 3.5 oz,Mexico
1,2,active sport boxer briefs - skinning up glory ...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",60 g 2.1 oz,Israel
2,3,active sport briefs - these superbreathable no...,"stretch mesh provides support, open-weave mesh...","4.6-oz 93% polyester 71% recycled /7% spandex,...",49 g 1.7 oz,Israel
3,4,"alpine guide pants - skin in, climb ice, switc...",durable stretch-woven polyester 47% recycled w...,7.6-oz 90-denier 92% polyester 47% recycled /8...,597 g 20.7 oz,Vietnam
4,5,"alpine wind jkt - on high ridges, steep ice an...","lightweight, breathable polyester ripstop fabr...",shell 1.3-oz 20-denier 100% polyester ripstop....,331 g 11.5 oz,China


In [4]:
missing_data_nb = data.isnull().sum()
missing_data_pct = data.isnull().sum() / len(data) * 100

df = pd.concat([missing_data_nb, missing_data_pct], axis=1, keys=['number', 'percentage'])
print(df[df['number'] > 0])

        number  percentage
weight       8         1.6
pays         8         1.6


In [5]:
data.nunique()

id                  500
description_text    488
details             421
fabric              227
weight              243
pays                 12
dtype: int64

In [6]:
data.shape

(500, 6)

In [7]:
text_columns = ["description_text", "details", "fabric", "weight", "pays"]

text = data[text_columns].fillna("").astype(str).agg(" ".join, axis=1)

In [8]:
display(text.head())
print("Text number :", len(text))

0    active classic boxers - there s a reason why o...
1    active sport boxer briefs - skinning up glory ...
2    active sport briefs - these superbreathable no...
3    alpine guide pants - skin in, climb ice, switc...
4    alpine wind jkt - on high ridges, steep ice an...
dtype: object

Text number : 500


In [9]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.95,
    min_df=2
)

tfidf_matrix = vectorizer.fit_transform(text)

print("Matrix dimensions :", tfidf_matrix.shape)
print("Tokens samples :", vectorizer.get_feature_names_out()[:20])

Matrix dimensions : (500, 2629)
Tokens samples : ['000' '10' '100' '103' '108' '109' '1096' '11' '112' '115' '118' '1188'
 '12' '1200' '121' '124' '126' '129' '13' '130']


In [10]:
pd.DataFrame(
    tfidf_matrix[:5].toarray(),
    columns=vectorizer.get_feature_names_out()
)

,000,10,100,103,108,109,1096,11,112,115,...,yoga,yoke,yosemite,zip,zipper,zippered,zippers,zipping,zips,zones
0,0.0,0.00000,0.082345,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
1,0.0,0.00000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
2,0.0,0.00000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
3,0.0,0.00000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.239127,0.073411,0.0,0.0,0.0
4,0.0,0.04785,0.047210,0.0,0.0,0.0,0.0,0.060475,0.0,0.0,...,0.0,0.0,0.0,0.0,0.089452,0.035530,0.054538,0.0,0.0,0.0


In [11]:
db = dbscan = DBSCAN(
    eps=0.5,
    min_samples=10,
    metric="cosine"
)

In [12]:
db.fit_predict(tfidf_matrix)

data["cluster"] = db.labels_

display(data["cluster"].value_counts().sort_index())

cluster
-1    420
 0     10
 1     55
 2     15
Name: count, dtype: int64

In [13]:
## Import go to build our own figure
import plotly.graph_objects as go

In [14]:
display(data.sort_values("cluster"))

,id,description_text,details,fabric,weight,pays,cluster
0,1,active classic boxers - there s a reason why o...,"silky capilene 1 fabric is ultralight, breatha...",3.7-oz 100% all-recycled polyester with gladio...,99 g 3.5 oz,Mexico,-1
325,326,thrift shorts - the unofficial short for those...,cool-wearing recycled polyester and organic co...,3.4-oz 65% all-recycled polyester/35% organic ...,207 g 7.2 oz,Thailand,-1
324,325,wavefarer board shorts - wavefarers won t bunc...,"nylon fabric is lightweight, durable and quick...",4.3-oz 100% nylon with dwr durable water repel...,121 g 4.2 oz,Thailand,-1
323,324,vitalti pants - soft and stretchy vitaliti pan...,plaited organic cotton and polyester stretch f...,5.7-oz 55% organic cotton/35% all-recycled pol...,377 g 13.1 oz,Thailand,-1
322,323,vitaliti vee - the stretch and refined drape o...,plaited organic cotton and all-recycled-polyes...,5.7-oz 55% organic cotton/35% all-recycled pol...,129 g 4.5 oz,Thailand,-1
...,...,...,...,...,...,...,...
86,87,merino 1 crew - in a day packed with decisions...,slow-washed merino wool resists odor naturally...,3.2-oz 108-g 63% 16.5 micron chlorine-free mer...,138 g 4.9 oz,USA,2
268,269,merino 2 t-shirt - our most versatile merino b...,slow-washed merino wool resists odor naturally...,4.85-oz 165-g 73% chlorine-free merino wool/27...,103 g 3.6 oz,USA,2
405,406,"merino 2 bottoms - a stop-and-go pace, like th...",slow-washed merino wool resists odor naturally...,4.85-oz 165-g 73% chlorine-free merino wool/27...,170 g 5.9 oz,USA,2
201,202,merino 2 crew - be moved by the people and cul...,slow-washed merino wool resists odor naturally...,4.85-oz 165-g 73% chlorine-free merino wool/27...,150 g 5.2 oz,USA,2


In [15]:
print(f"clusters number : {len(set(data['cluster'])) - 1}")  # On soustrait 1 pour exclure le bruit (-1)

clusters number : 3


Le cluster -1 est le cluster considéré comme du bruit par l'algorithme Dbscan. Nous allons tenter de réduire le nombre de produits dans ce cluster en ajustant les paramètres de l'algorithme.

In [16]:
parameters = {
    "eps": [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "min_samples": [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15]
}

results = []

for params in ParameterGrid(parameters):
    clusters = DBSCAN(**params, metric="cosine").fit_predict(tfidf_matrix)
    without_noise = clusters != -1
    try : 
        score = silhouette_score(
            tfidf_matrix[without_noise],
            clusters[without_noise],
            metric="cosine"
        )
    except ValueError:
        score = np.nan

    results.append({
        **params,
        "silhouette": score,
        "noise_pct": (1 - without_noise.mean()) * 100,
        "clusters_nb": len(set(clusters)) - (1 if -1 in clusters else 0)
    })

results = pd.DataFrame(results)
# display(results.sort_values("clusters_nb", ascending=False))

In [17]:
# candidates = results[(results["clusters_nb"] <= 20) & (results["clusters_nb"] >= 10) & results["silhouette"].notna()]

In [21]:
candidates = results[(results["clusters_nb"].between(10, 20)) & (results["noise_pct"] < 30)].sort_values("silhouette", ascending=False)
candidates

,eps,min_samples,silhouette,noise_pct,clusters_nb
39,0.7,8,0.188290,23.8,10
38,0.7,7,0.168158,16.8,11
36,0.7,5,0.162444,11.8,12
37,0.7,6,0.160537,14.0,12
35,0.7,4,0.154133,7.2,17
34,0.7,3,0.144693,4.8,18


In [ ]:

best = candidates.sort_values("silhouette", ascending=False).iloc[0]
display(best)

eps             0.70000
min_samples     8.00000
silhouette      0.18829
noise_pct      23.80000
clusters_nb    10.00000
Name: 39, dtype: float64

In [ ]:
# import numpy as np
# import plotly.graph_objects as go
fig = go.Figure()

# Loop through each label for our cluster
for i in np.unique(db.labels_):
    label = tfidf_matrix[db.labels_ == i]
    fig.add_trace(go.Scatter(x=results["noise_pct"], y=results["silhouette"], mode="markers", name="Cluster {}".format(i)))
    fig.update_layout(
    xaxis_title="Noise percentage)",
    yaxis_title="Score de silhouette")
fig.show()

Le code trace les mêmes points (results["noise_pct"], results["silhouette"]) à chaque itération de la boucle : label n’est jamais utilisé, donc tous les “clusters” sont identiques et superposés. Ces données de results décrivent les essais de paramètres DBSCAN, pas les clusters.

Pour visualiser les clusters, il faut réduire tfidf_matrix en 2D, par exemple :

In [24]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import TruncatedSVD

coords = TruncatedSVD(n_components=2, random_state=42).fit_transform(tfidf_matrix)

fig = go.Figure()
for i in np.unique(db.labels_):
    mask = db.labels_ == i
    fig.add_trace(go.Scatter(x=coords[mask, 0], y=coords[mask, 1], mode="markers", name=f"Cluster {i}"))
fig.show()

# 